# B3NS — Selective negative prompting (SNP)

## Why
- **B3N:** negative points break merged masks (median S6 guided-mask area 26,609 → 1,871 px) and give the first **significant** S6 gain over H1b (266.9 vs 275.0), but fragment sparse objects (S1 19.2).
- **B3L §5:** SAM2.1-Large + masked input (M0) gives the best S1–S4 profile (H1b-L[M0]: S1 6.89, S2 6.14, S3 9.93) but loses dense objects.
- Negative points are only needed where a mask is **merged**, and **rule A** already detects merged masks at mask level (units recall 0.927 in B3GK). No image-level gate is needed.

## Pre-declared design (before any B3NS result)
| Step | Rule |
|---|---|
| Base mask set | GPV_R4 from the base's guided + B0 masks (NMS 0.5, B0 pool) |
| Flag | rule A on the R4 set: area > 4 × median kept area AND ≥ 2 frozen D1 peaks |
| Re-prompt | every `dense_recovery` prompt inside a flagged mask → positive + negatives (up to 4 nearest other prompts within 2 × the image's median nearest-neighbour prompt distance; all image prompts considered) |
| Keep | a new mask is kept only if > 50% of it lies inside the flagged mask it replaces |
| Replace | flagged masks with ≥ 1 new mask are replaced by them; flagged masks with no prompt inside stay; then IoU-NMS 0.5 over the whole set |
| Count | **GPV_R4-SN** = masks; **H1b-SN** = H1b on the new set (rule A and local recovery recomputed) |
| Bases | **[S]** SAM2.1-Small, full image (frozen caches); **[LM0]** SAM2.1-Large, masked input M0 (B3L §5 caches; re-prompting with Large on the masked image) |
| Targets | T1 (S1, S2 vs B0_v1), T2′ (S5, S6 vs H1b_v1), T3′ (reweighted MAE vs H1b_v1) |

## Run order
T4 → 1.0–1.2 → 2.1–2.2 → 3.1 (resumable) → 4.1.

# 1. Setup

In [1]:
# 1.0 — Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# ============================================================
# Step 1.1 — Paths, repo modules, train pool, frozen-system counts
# ============================================================

import importlib, json, subprocess, sys, time
from datetime import datetime, timezone
from pathlib import Path
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

PROJECT_ROOT = Path("/content/drive/MyDrive/MSc_Project_ZeroShot_Counting")
RESULTS_ROOT = PROJECT_ROOT / "results"
FSC147_RAW = PROJECT_ROOT / "datasets" / "FSC147" / "raw"
FSC147_IMAGE_DIR = FSC147_RAW / "images_384_VarV2"
TD_ROOT = PROJECT_ROOT / "cache" / "FSC147_train_dev400"
B3S_DIR = RESULTS_ROOT / "B3S_Semantic_Improvement"
OUT_DIR = RESULTS_ROOT / "B3NS_Selective_Negative_Prompts"
OUT_DIR.mkdir(parents=True, exist_ok=True)

REPO_URL = "https://github.com/paulmcchan/Surrey-MScProject-Zero-Shot-Object-Counting.git"
REPO_DIR = Path("/content/Surrey-MScProject-Zero-Shot-Object-Counting")
!rm -rf "{REPO_DIR}"
!git clone -q "{REPO_URL}" "{REPO_DIR}"
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
REPO_COMMIT = subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip()

import src.pipeline.io_cache as io_cache
import src.pipeline.sg as sg
import src.pipeline.gpv as gpv
import src.pipeline.hybrid as hybrid
import src.pipeline.b0 as b0
for m in (io_cache, sg, gpv, hybrid, b0):
    importlib.reload(m)

STRATA = ["S1_0_20", "S2_20_40", "S3_40_60", "S4_60_80", "S5_80_95", "S6_95_100"]
W = {"S1_0_20": .20, "S2_20_40": .20, "S3_40_60": .20, "S4_60_80": .20, "S5_80_95": .15, "S6_95_100": .05}
TD_DIRS = {"b0_masks": TD_ROOT / "b0_masks", "a3c_masks": TD_ROOT / "a3c_masks",
           "depth": TD_ROOT / "depth", "m1_split_labels": TD_ROOT / "m1_split_labels"}
SG_CFG = sg.SGConfig.from_frozen_json(RESULTS_ROOT / "B3R_SG_Rebuild_and_Fixes" / "SG_v1.1_frozen_config.json")

td_df = pd.read_csv(TD_ROOT / "FSC147_train_dev400.csv")
td_res = pd.read_csv(B3S_DIR / "B3S_step1_per_image.csv")
assert len(td_df) == 400 and len(td_res) == 400
print("Repo commit:", REPO_COMMIT, "| output:", OUT_DIR)

Repo commit: d70bb1d6d631c12156ba7bf1881d22f5b2eb6100 | output: /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B3NS_Selective_Negative_Prompts


In [3]:
import torch

# ============================================================
# Step 1.2 — Shared helpers: guided masks, B0 meta filtering, systems, targets
# ============================================================

def guided_masks_for_prompts(predictor, image_np, prompts_xy):
    h, w = image_np.shape[:2]
    masks, scores = [], []
    if len(prompts_xy):
        with torch.inference_mode():
            predictor.set_image(image_np)
            for p in prompts_xy:
                m, s, _ = predictor.predict(point_coords=np.asarray([p], dtype=np.float32),
                                            point_labels=np.asarray([1], dtype=np.int32),
                                            multimask_output=False)
                masks.append(np.asarray(m[0], dtype=bool)); scores.append(float(np.asarray(s)[0]))
    return (np.asarray(masks) if masks else np.zeros((0, h, w), bool)), np.asarray(scores, float)

def filter_meta(meta, keep):
    return {k: np.asarray(v)[keep] for k, v in meta.items()}

def systems_from_masks(b0_masks, b0_iou, g_masks, g_scores, a3c, depth, san_target):
    keep = gpv.nms_masks(g_masks, g_scores, 0.5) if g_masks.shape[0] else np.zeros(0, int)
    r4_m, _, _ = gpv.build_r4(g_masks[keep], np.asarray(g_scores)[keep], b0_masks, b0_iou, 0.5)
    d1 = gpv.d1_peaks(san_target)
    h1b = float(hybrid.h1b_count(r4_m, d1, a3c, depth, SG_CFG)[0])
    return {"B0": int(b0_masks.shape[0]), "GPV_R4": int(r4_m.shape[0]), "H1b": h1b}

def evaluate(frame, systems, label):
    gt = frame["gt_count"].to_numpy(float); st = frame["density_stratum"].to_numpy()
    AE = {s: np.abs(frame[s].to_numpy(float) - gt) for s in systems}
    IDX = {}
    for j, k in enumerate(STRATA):
        pos = np.flatnonzero(st == k)
        IDX[k] = pos[np.random.default_rng(1000 + j).integers(0, len(pos), size=(10_000, len(pos)))]
    def sci(a, b, k):
        d = AE[a][IDX[k]].mean(1) - AE[b][IDX[k]].mean(1)
        return AE[a][st == k].mean() - AE[b][st == k].mean(), np.percentile(d, 2.5), np.percentile(d, 97.5)
    def rci(a, b):
        d = sum(w * (AE[a][IDX[k]].mean(1) - AE[b][IDX[k]].mean(1)) for k, w in W.items())
        o = sum(w * (AE[a][st == k].mean() - AE[b][st == k].mean()) for k, w in W.items())
        return o, np.percentile(d, 2.5), np.percentile(d, 97.5)
    res = pd.DataFrame([{"system": s,
                         "MAE_reweighted": sum(w * AE[s][st == k].mean() for k, w in W.items()),
                         "mean_signed": float((frame[s] - frame["gt_count"]).mean()),
                         **{f"MAE_{k[:2]}": AE[s][st == k].mean() for k in STRATA}} for s in systems])
    rows = []
    for s in [x for x in systems if x not in ("B0_v1", "GPV_R4_v1", "H1b_v1")]:
        rec = {"system": s}; t1 = []; t2 = []
        for k in ["S1_0_20", "S2_20_40"]:
            d, lo, hi = sci(s, "B0_v1", k); rec[f"{k[:2]} vs B0"] = f"{d:+.2f} [{lo:+.2f}, {hi:+.2f}]"; t1.append(lo <= 0)
        for k in ["S5_80_95", "S6_95_100"]:
            d, lo, hi = sci(s, "H1b_v1", k); rec[f"{k[:2]} vs H1b"] = f"{d:+.2f} [{lo:+.2f}, {hi:+.2f}]"; t2.append(lo <= 0)
        d, lo, hi = rci(s, "H1b_v1"); rec["overall vs H1b"] = f"{d:+.2f} [{lo:+.2f}, {hi:+.2f}]"
        rec.update({"T1": all(t1), "T2'": all(t2), "T3'": hi < 0, "ALL": all(t1) and all(t2) and hi < 0})
        rows.append(rec)
    print(f"{label} — metrics:"); display(res.round(2))
    print("Targets (difference [95% CI]; negative = system better):"); display(pd.DataFrame(rows))
    return res, pd.DataFrame(rows)
print("helpers ready")

helpers ready


# 2. SAM2.1-Small and SAM2.1-Large predictors

In [5]:
# ============================================================
# Step 2.1 — Install SAM2 (frozen commit)
# ============================================================

import torch
assert torch.cuda.is_available(), "Switch the runtime to T4."
DEVICE = "cuda"
%cd /content
if not Path("/content/sam2_repo/sam2/__init__.py").exists():
    !git clone -q https://github.com/facebookresearch/sam2.git /content/sam2_repo
    !git -C /content/sam2_repo checkout -q {b0.FROZEN_SAM2_COMMIT}
    %cd /content/sam2_repo
    !pip install -q -e .
    %cd /content
if "/content/sam2_repo" not in sys.path:
    sys.path.insert(0, "/content/sam2_repo")
importlib.invalidate_caches()
from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor
print("SAM2 commit:", subprocess.check_output(["git", "-C", "/content/sam2_repo", "rev-parse", "HEAD"], text=True).strip())
print("GPU:", torch.cuda.get_device_name(0), "| torch", torch.__version__, "| CUDA", torch.version.cuda)

/content
/content/sam2_repo
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 3.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.8/168.8 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 11.6 MB/s eta 0:00:00
  Building editable for SAM-2 (pyproject.toml) ... done
/content
SAM2 commit: 2b90b9f5ceec907a1c18123530e92e794ad901a4
GPU: Tesla T4 | torch 2.11.0+cu130 | CUDA 13.0


In [6]:
# ============================================================
# Step 2.2 — Build both predictors
# ============================================================

SAM2_S = PROJECT_ROOT / "Checkpoints" / "SAM2" / "sam2.1_hiera_small.pt"
SAM2_L = PROJECT_ROOT / "Checkpoints" / "SAM2" / "sam2.1_hiera_large.pt"
assert b0.sha256_file(SAM2_S) == b0.FROZEN_CHECKPOINT_SHA256
assert SAM2_L.exists(), "Run B3L Step 2.2 once to download SAM2.1-Large."
predictor_S = SAM2ImagePredictor(build_sam2(b0.FROZEN_MODEL_CFG, str(SAM2_S), device=DEVICE).eval())
predictor_L = SAM2ImagePredictor(build_sam2("configs/sam2.1/sam2.1_hiera_l.yaml", str(SAM2_L), device=DEVICE).eval())
print("Predictors ready: Small (frozen) and Large")

Predictors ready: Small (frozen) and Large


# 3. Selective negative prompting on both bases

In [ ]:
# ============================================================
# Step 3.1 — SNP for bases [S] and [LM0] (resumable, per-image JSON)
# ============================================================

from scipy.spatial import cKDTree
MAX_NEG, RADIUS_FACTOR = 4, 2.0
TD_TILED = TD_ROOT / "san_vocab_tiled"
LM_ROOT = TD_ROOT / "sam2_large_masked_M0"
ROWS = OUT_DIR / "rows"
ROWS.mkdir(parents=True, exist_ok=True)

def negative_sets(prompts):
    # Same rule as B3N: up to MAX_NEG nearest other prompts within RADIUS_FACTOR x median NN distance
    n = len(prompts)
    if n < 2:
        return [np.zeros((0, 2))] * n
    tree = cKDTree(prompts)
    dist, idx = tree.query(prompts, k=min(n, MAX_NEG + 1))
    d_med = float(np.median(dist[:, 1]))
    out = []
    for i in range(n):
        sel = [j for d, j in zip(dist[i, 1:], idx[i, 1:]) if d <= RADIUS_FACTOR * d_med]
        out.append(prompts[sel] if sel else np.zeros((0, 2)))
    return out

def snp(predictor, image_np, r4_m, r4_s, prompts, d1):
    # Returns the new mask set, scores and diagnostics
    n = r4_m.shape[0]
    if n == 0 or len(prompts) == 0:
        return r4_m, r4_s, {"flagged": 0, "reprompted": 0, "replaced": 0, "new_masks": 0}
    merged = gpv.merged_mask_flags(r4_m, gpv.points_per_mask(r4_m, d1), "A")
    if not merged.any():
        return r4_m, r4_s, {"flagged": 0, "reprompted": 0, "replaced": 0, "new_masks": 0}
    h, w = r4_m.shape[1:]
    px = np.clip(np.rint(prompts[:, 0]).astype(int), 0, w - 1)
    py = np.clip(np.rint(prompts[:, 1]).astype(int), 0, h - 1)
    negs = negative_sets(prompts)
    keep_idx, new_m, new_s = [], [], []
    n_rep = n_replaced = 0
    with torch.inference_mode():
        predictor.set_image(image_np)
        for i in range(n):
            if not merged[i]:
                keep_idx.append(i); continue
            inside = np.flatnonzero(r4_m[i][py, px])
            got = []
            for k in inside:
                coords = np.vstack([prompts[k][None], negs[k]]).astype(np.float32)
                labels = np.array([1] + [0] * len(negs[k]), dtype=np.int32)
                m, s, _ = predictor.predict(point_coords=coords, point_labels=labels, multimask_output=False)
                m = np.asarray(m[0], dtype=bool); n_rep += 1
                if m.sum() and (m & r4_m[i]).sum() > 0.5 * m.sum():
                    got.append((m, float(np.asarray(s)[0])))
            if got:
                n_replaced += 1
                new_m += [g[0] for g in got]; new_s += [g[1] for g in got]
            else:
                keep_idx.append(i)
    pool = np.concatenate([r4_m[keep_idx], np.asarray(new_m, bool).reshape(-1, h, w)])
    scores = np.concatenate([np.asarray(r4_s)[keep_idx], np.asarray(new_s, float)])
    kk = gpv.nms_masks(pool, scores, 0.5)
    return pool[kk], scores[kk], {"flagged": int(merged.sum()), "reprompted": n_rep,
                                  "replaced": n_replaced, "new_masks": len(new_m)}

todo = [r for r in td_df.itertuples(index=False) if not (ROWS / f"{int(r.dataset_index):05d}.json").exists()]
print(f"Done: {len(td_df) - len(todo)} | to process: {len(todo)}")
t0 = time.time()
for i, r in enumerate(tqdm(todo, desc="SNP")):
    di, stem = r.dataset_index, io_cache.image_stem(r.image_id)
    p = io_cache.cache_paths(di, r.image_id, TD_DIRS)
    a3c, _ = io_cache.load_a3c_mask(p["a3c_masks"])
    dep = io_cache.load_depth(p["depth"])
    san = io_cache.load_san_scores(TD_ROOT / "san" / io_cache.san_scores_filename(r.image_id))
    d1 = gpv.d1_peaks(san["target_score"])
    g0 = io_cache.load_guided_masks(TD_ROOT / "guided_dense_recovery" / f"{int(di):04d}_{stem}.npz")
    prompts = g0["prompts_xy"]
    img = b0.load_image_rgb_cv2(FSC147_IMAGE_DIR / r.image_id)
    row = {"dataset_index": int(di)}

    # ---- base [S]: Small, full image (frozen caches)
    b0m, b0meta = io_cache.load_b0_masks(p["b0_masks"])
    kg = gpv.nms_masks(g0["masks"], g0["scores"], 0.5)
    r4_m, r4_s, _ = gpv.build_r4(g0["masks"][kg], g0["scores"][kg], b0m, b0meta["predicted_iou"], 0.5)
    sm, ss, dg = snp(predictor_S, img, r4_m, r4_s, prompts, d1)
    row.update({"GPV_R4-SN[S]": int(sm.shape[0]),
                "H1b-SN[S]": float(hybrid.h1b_count(sm, d1, a3c, dep, SG_CFG)[0]),
                **{f"{k}[S]": v for k, v in dg.items()}})

    # ---- base [LM0]: Large, masked input M0 (B3L Section 5 caches)
    with np.load(TD_TILED / f"{stem}.npz") as v:
        reg = v["tiled_argmax"] == 0
    masked = (img * reg[..., None]).astype(np.uint8)
    bL, bLmeta = io_cache.load_b0_masks(LM_ROOT / "b0" / f"{int(di):05d}_{stem}.npz")
    gL = io_cache.load_guided_masks(LM_ROOT / "guided" / f"{int(di):04d}_{stem}.npz")
    kgL = gpv.nms_masks(gL["masks"], gL["scores"], 0.5) if gL["masks"].shape[0] else np.zeros(0, int)
    r4L_m, r4L_s, _ = gpv.build_r4(gL["masks"][kgL], np.asarray(gL["scores"])[kgL], bL, bLmeta["predicted_iou"], 0.5)
    lm_, ls_, dgL = snp(predictor_L, masked, r4L_m, r4L_s, prompts, d1)
    row.update({"GPV_R4-SN[LM0]": int(lm_.shape[0]),
                "H1b-SN[LM0]": float(hybrid.h1b_count(lm_, d1, a3c, dep, SG_CFG)[0]),
                **{f"{k}[LM0]": v for k, v in dgL.items()}})
    with open(ROWS / f"{int(di):05d}.json", "w") as f:
        json.dump(row, f)
    if (i + 1) % 25 == 0:
        torch.cuda.empty_cache()
print(f"Elapsed: {(time.time() - t0) / 60:.1f} min")

Done: 0 | to process: 400


SNP:   0%|          | 0/400 [00:00<?, ?it/s]

Elapsed: 37.4 min


# 4. Results and targets

In [ ]:
# ============================================================
# Step 4.1 — Diagnostics, metrics and targets
# ============================================================

ROWS = OUT_DIR / "rows"
sn = pd.DataFrame([json.load(open(p)) for p in sorted(ROWS.glob("*.json"))])
assert len(sn) == 400, len(sn)
lm_prev = pd.read_csv(RESULTS_ROOT / "B3L_SAM2_Large" / "section5_large_masked" / "B3L_s5_per_image.csv")[
    ["dataset_index", "GPV_R4-L[M0]", "H1b-L[M0]"]]
neg_prev = pd.read_csv(RESULTS_ROOT / "B3N_Negative_Point_Prompts" / "B3N_per_image.csv")[
    ["dataset_index", "H1b[neg]"]]
frame = (sn.merge(lm_prev, on="dataset_index").merge(neg_prev, on="dataset_index")
         .merge(td_res[["dataset_index", "gt_count", "density_stratum", "B0", "GPV_R4_v1", "H1b_v1"]]
                .rename(columns={"B0": "B0_v1"}), on="dataset_index", validate="one_to_one"))

print("Flagged masks, re-prompts and replacements per image, by stratum:")
diag_cols = [c for c in frame.columns if c.split("[")[0] in ("flagged", "reprompted", "replaced", "new_masks")]
display(frame.groupby("density_stratum")[diag_cols].mean().reindex(STRATA).round(2))

SYS = ["B0_v1", "GPV_R4_v1", "H1b_v1", "H1b[neg]", "GPV_R4-L[M0]", "H1b-L[M0]",
       "GPV_R4-SN[S]", "H1b-SN[S]", "GPV_R4-SN[LM0]", "H1b-SN[LM0]"]
res, targets = evaluate(frame, SYS, "Selective negative prompting (train_dev400)")

frame.to_csv(OUT_DIR / "B3NS_per_image.csv", index=False)
res.to_csv(OUT_DIR / "B3NS_metrics.csv", index=False)
targets.to_csv(OUT_DIR / "B3NS_targets.csv", index=False)
with open(OUT_DIR / "B3NS_metadata.json", "w") as f:
    json.dump({"notebook": "B3NS_Selective_Negative_Prompts",
               "rule": "re-prompt prompts inside rule-A merged masks with positive + up to 4 nearest negatives "
                       "(within 2 x median NN prompt distance); keep new masks > 50% inside the flagged mask; "
                       "replace flagged masks that got >= 1 new mask; NMS 0.5",
               "bases": {"S": "SAM2.1-Small, full image (frozen caches)",
                         "LM0": "SAM2.1-Large, masked input M0 (B3L section 5 caches)"},
               "date_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
               "repo_commit": REPO_COMMIT, "heldout_accessed": False, "test_set_accessed": False}, f, indent=2)
print("Saved to:", OUT_DIR)

Flagged masks, re-prompts and replacements per image, by stratum:


,flagged[S],reprompted[S],replaced[S],new_masks[S],flagged[LM0],reprompted[LM0],replaced[LM0],new_masks[LM0]
density_stratum,,,,,,,,
S1_0_20,1.20,12.84,1.19,12.12,0.80,10.46,0.80,10.35
S2_20_40,0.94,11.56,0.92,11.06,0.76,12.21,0.72,11.79
S3_40_60,1.18,32.32,1.17,30.75,0.92,20.85,0.92,19.52
S4_60_80,1.42,17.17,1.37,16.65,1.28,12.40,1.27,12.03
S5_80_95,1.68,24.58,1.68,23.58,1.78,26.67,1.78,26.28
S6_95_100,2.27,86.68,2.25,84.28,2.00,56.18,1.97,54.87


Selective negative prompting (train_dev400) — metrics:


,system,MAE_reweighted,mean_signed,MAE_S1,MAE_S2,MAE_S3,MAE_S4,MAE_S5,MAE_S6
0,B0_v1,35.67,-62.67,5.56,7.74,13.30,22.93,59.03,338.08
1,GPV_R4_v1,29.94,-46.78,8.62,7.70,12.45,16.13,40.52,297.63
2,H1b_v1,28.65,-41.28,10.46,9.49,12.35,14.20,37.33,274.98
3,H1b[neg],36.95,-30.90,19.19,19.56,27.60,22.87,38.42,266.87
4,GPV_R4-L[M0],30.36,-54.12,5.35,5.28,10.23,16.07,47.28,317.70
5,H1b-L[M0],28.91,-49.22,6.89,6.14,9.93,14.90,42.42,299.43
6,GPV_R4-SN[S],32.03,-39.46,12.30,10.07,19.27,19.50,39.35,277.88
7,H1b-SN[S],31.45,-37.94,12.80,10.95,19.00,18.18,37.97,271.28
8,GPV_R4-SN[LM0],31.30,-47.27,9.07,8.39,14.90,15.45,45.52,298.25
9,H1b-SN[LM0],30.47,-45.25,9.56,8.59,15.08,15.00,42.35,289.43


Targets (difference [95% CI]; negative = system better):


,system,S1 vs B0,S2 vs B0,S5 vs H1b,S6 vs H1b,overall vs H1b,T1,T2',T3',ALL
0,H1b[neg],"+13.62 [+10.64, +16.70]","+11.82 [+8.64, +15.16]","+1.08 [-3.17, +5.62]","-8.12 [-12.67, -3.15]","+8.30 [+6.76, +9.91]",False,True,False,False
1,GPV_R4-L[M0],"-0.21 [-1.59, +1.11]","-2.46 [-3.99, -0.89]","+9.95 [+3.22, +18.33]","+42.72 [+29.13, +57.53]","+1.71 [+0.17, +3.37]",True,False,False,False
2,H1b-L[M0],"+1.33 [-0.44, +3.11]","-1.60 [-3.34, +0.16]","+5.08 [-0.08, +11.52]","+24.45 [+12.60, +38.02]","+0.26 [-1.03, +1.64]",True,False,False,False
3,GPV_R4-SN[S],"+6.74 [+4.16, +9.62]","+2.34 [+0.04, +4.89]","+2.02 [-2.10, +7.40]","+2.90 [-3.88, +10.50]","+3.38 [+2.11, +4.76]",False,True,False,False
4,H1b-SN[S],"+7.24 [+4.68, +10.02]","+3.21 [+0.73, +5.88]","+0.63 [-2.70, +4.43]","-3.70 [-9.48, +2.58]","+2.80 [+1.69, +3.98]",False,True,False,False
5,GPV_R4-SN[LM0],"+3.51 [+0.97, +6.24]","+0.65 [-1.81, +3.25]","+8.18 [+1.67, +16.10]","+23.27 [+9.95, +37.92]","+2.65 [+1.06, +4.33]",False,False,False,False
6,H1b-SN[LM0],"+4.00 [+1.35, +6.83]","+0.85 [-1.71, +3.50]","+5.02 [-0.40, +11.70]","+14.45 [+1.45, +28.95]","+1.82 [+0.37, +3.37]",False,False,False,False


Saved to: /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B3NS_Selective_Negative_Prompts


## Sections 3–4 result (SNP as first designed)

Selective negatives halved the sparse damage of all-prompt negatives but kept little of the S6 gain (H1b-SN[S]: S1 12.80, S3 19.00, S6 271.3 n.s.; reweighted 31.45; H1b-SN[LM0] 30.47). **Cause:** rule A flags ~1 mask per sparse image, but each held ~12 prompts (S3: ~32) already covered by individual masks; re-prompting them created fragments of objects already counted, surviving NMS (IoU < 0.5). The region definition used by GPV_K / H1b (merged mask minus individually covered pixels) was not applied.

# 5. SNP-v2 — re-prompt only uncovered prompts

## Pre-declared correction (no new parameter)
- **Region** of a flagged mask = the flagged mask minus pixels of every **unflagged** kept mask, minus regions already assigned to higher-scoring flagged masks (the GPV_K / H1b region definition).
- **Re-prompt** only `dense_recovery` prompts lying **inside the region** (positive + the same B3N negatives).
- **Keep** a new mask only if > 50% of it lies inside the **region**.
- Replace / keep / NMS and counting exactly as Section 3; both bases [S] and [LM0].

**Run order:** T4 → 1.0–1.2 → 2.1–2.2 → 5.1 → 5.2 (Sections 3–4 not needed).

In [7]:
# ============================================================
# Step 5.1 — SNP-v2 on bases [S] and [LM0] (resumable)
# ============================================================

from scipy.spatial import cKDTree
MAX_NEG, RADIUS_FACTOR = 4, 2.0
TD_TILED = TD_ROOT / "san_vocab_tiled"
LM_ROOT = TD_ROOT / "sam2_large_masked_M0"
ROWS2 = OUT_DIR / "rows_v2"
ROWS2.mkdir(parents=True, exist_ok=True)

def negative_sets(prompts):
    n = len(prompts)
    if n < 2:
        return [np.zeros((0, 2))] * n
    tree = cKDTree(prompts)
    dist, idx = tree.query(prompts, k=min(n, MAX_NEG + 1))
    d_med = float(np.median(dist[:, 1]))
    out = []
    for i in range(n):
        sel = [j for d, j in zip(dist[i, 1:], idx[i, 1:]) if d <= RADIUS_FACTOR * d_med]
        out.append(prompts[sel] if sel else np.zeros((0, 2)))
    return out

def snp_v2(predictor, image_np, r4_m, r4_s, prompts, d1):
    n = r4_m.shape[0]
    empty = {"flagged": 0, "reprompted": 0, "replaced": 0, "new_masks": 0}
    if n == 0 or len(prompts) == 0:
        return r4_m, r4_s, empty
    merged = gpv.merged_mask_flags(r4_m, gpv.points_per_mask(r4_m, d1), "A")
    if not merged.any():
        return r4_m, r4_s, empty
    h, w = r4_m.shape[1:]
    taken = r4_m[~merged].any(axis=0) if (~merged).any() else np.zeros((h, w), bool)
    px = np.clip(np.rint(prompts[:, 0]).astype(int), 0, w - 1)
    py = np.clip(np.rint(prompts[:, 1]).astype(int), 0, h - 1)
    negs = negative_sets(prompts)
    order = [i for i in np.argsort(-np.asarray(r4_s), kind="stable") if merged[i]]
    replaced_idx, new_m, new_s = set(), [], []
    n_rep = 0
    with torch.inference_mode():
        predictor.set_image(image_np)
        for i in order:
            region = r4_m[i] & ~taken
            taken |= region
            inside = np.flatnonzero(region[py, px])
            got = []
            for k in inside:
                coords = np.vstack([prompts[k][None], negs[k]]).astype(np.float32)
                labels = np.array([1] + [0] * len(negs[k]), dtype=np.int32)
                m, s, _ = predictor.predict(point_coords=coords, point_labels=labels, multimask_output=False)
                m = np.asarray(m[0], dtype=bool); n_rep += 1
                if m.sum() and (m & region).sum() > 0.5 * m.sum():
                    got.append((m, float(np.asarray(s)[0])))
            if got:
                replaced_idx.add(i)
                new_m += [g[0] for g in got]; new_s += [g[1] for g in got]
    keep_idx = [i for i in range(n) if i not in replaced_idx]
    pool = np.concatenate([r4_m[keep_idx], np.asarray(new_m, bool).reshape(-1, h, w)])
    scores = np.concatenate([np.asarray(r4_s)[keep_idx], np.asarray(new_s, float)])
    kk = gpv.nms_masks(pool, scores, 0.5)
    return pool[kk], scores[kk], {"flagged": int(merged.sum()), "reprompted": n_rep,
                                  "replaced": len(replaced_idx), "new_masks": len(new_m)}

todo = [r for r in td_df.itertuples(index=False) if not (ROWS2 / f"{int(r.dataset_index):05d}.json").exists()]
print(f"Done: {len(td_df) - len(todo)} | to process: {len(todo)}")
t0 = time.time()
for i, r in enumerate(tqdm(todo, desc="SNP-v2")):
    di, stem = r.dataset_index, io_cache.image_stem(r.image_id)
    p = io_cache.cache_paths(di, r.image_id, TD_DIRS)
    a3c, _ = io_cache.load_a3c_mask(p["a3c_masks"])
    dep = io_cache.load_depth(p["depth"])
    san = io_cache.load_san_scores(TD_ROOT / "san" / io_cache.san_scores_filename(r.image_id))
    d1 = gpv.d1_peaks(san["target_score"])
    g0 = io_cache.load_guided_masks(TD_ROOT / "guided_dense_recovery" / f"{int(di):04d}_{stem}.npz")
    prompts = g0["prompts_xy"]
    img = b0.load_image_rgb_cv2(FSC147_IMAGE_DIR / r.image_id)
    row = {"dataset_index": int(di)}

    b0m, b0meta = io_cache.load_b0_masks(p["b0_masks"])
    kg = gpv.nms_masks(g0["masks"], g0["scores"], 0.5)
    r4_m, r4_s, _ = gpv.build_r4(g0["masks"][kg], g0["scores"][kg], b0m, b0meta["predicted_iou"], 0.5)
    sm, ss, dg = snp_v2(predictor_S, img, r4_m, r4_s, prompts, d1)
    row.update({"GPV_R4-SN2[S]": int(sm.shape[0]),
                "H1b-SN2[S]": float(hybrid.h1b_count(sm, d1, a3c, dep, SG_CFG)[0]),
                **{f"{k}2[S]": v for k, v in dg.items()}})

    with np.load(TD_TILED / f"{stem}.npz") as v:
        reg = v["tiled_argmax"] == 0
    masked = (img * reg[..., None]).astype(np.uint8)
    bL, bLmeta = io_cache.load_b0_masks(LM_ROOT / "b0" / f"{int(di):05d}_{stem}.npz")
    gL = io_cache.load_guided_masks(LM_ROOT / "guided" / f"{int(di):04d}_{stem}.npz")
    kgL = gpv.nms_masks(gL["masks"], gL["scores"], 0.5) if gL["masks"].shape[0] else np.zeros(0, int)
    r4L_m, r4L_s, _ = gpv.build_r4(gL["masks"][kgL], np.asarray(gL["scores"])[kgL], bL, bLmeta["predicted_iou"], 0.5)
    lm_, ls_, dgL = snp_v2(predictor_L, masked, r4L_m, r4L_s, prompts, d1)
    row.update({"GPV_R4-SN2[LM0]": int(lm_.shape[0]),
                "H1b-SN2[LM0]": float(hybrid.h1b_count(lm_, d1, a3c, dep, SG_CFG)[0]),
                **{f"{k}2[LM0]": v for k, v in dgL.items()}})
    with open(ROWS2 / f"{int(di):05d}.json", "w") as f:
        json.dump(row, f)
    if (i + 1) % 25 == 0:
        torch.cuda.empty_cache()
print(f"Elapsed: {(time.time() - t0) / 60:.1f} min")

Done: 0 | to process: 400


SNP-v2:   0%|          | 0/400 [00:00<?, ?it/s]

Elapsed: 70.2 min


In [8]:
# ============================================================
# Step 5.2 — SNP-v2 diagnostics, metrics and targets
# ============================================================

ROWS2 = OUT_DIR / "rows_v2"
s2 = pd.DataFrame([json.load(open(p)) for p in sorted(ROWS2.glob("*.json"))])
assert len(s2) == 400, len(s2)
v1 = pd.read_csv(OUT_DIR / "B3NS_per_image.csv")[["dataset_index", "H1b-SN[S]", "H1b-SN[LM0]", "H1b[neg]", "H1b-L[M0]"]] \
    if (OUT_DIR / "B3NS_per_image.csv").exists() else None
frame2 = s2.merge(td_res[["dataset_index", "gt_count", "density_stratum", "B0", "GPV_R4_v1", "H1b_v1"]]
                  .rename(columns={"B0": "B0_v1"}), on="dataset_index", validate="one_to_one")
if v1 is not None:
    frame2 = frame2.merge(v1, on="dataset_index")

print("SNP-v2: flagged masks, re-prompts and replacements per image, by stratum:")
dcols = [c for c in frame2.columns if c.split("2[")[0] in ("flagged", "reprompted", "replaced", "new_masks")]
display(frame2.groupby("density_stratum")[dcols].mean().reindex(STRATA).round(2))

SYS2 = ["B0_v1", "GPV_R4_v1", "H1b_v1"] + \
       ([c for c in ["H1b[neg]", "H1b-L[M0]", "H1b-SN[S]", "H1b-SN[LM0]"] if c in frame2.columns]) + \
       ["GPV_R4-SN2[S]", "H1b-SN2[S]", "GPV_R4-SN2[LM0]", "H1b-SN2[LM0]"]
res2, targets2 = evaluate(frame2, SYS2, "SNP-v2 (train_dev400)")

D5 = OUT_DIR / "section5_snp_v2"
D5.mkdir(parents=True, exist_ok=True)
frame2.to_csv(D5 / "B3NS_s5_per_image.csv", index=False)
res2.to_csv(D5 / "B3NS_s5_metrics.csv", index=False)
targets2.to_csv(D5 / "B3NS_s5_targets.csv", index=False)
with open(D5 / "B3NS_s5_metadata.json", "w") as f:
    json.dump({"section": "5 — SNP-v2 (uncovered prompts only)",
               "region": "flagged mask minus unflagged kept masks minus earlier flagged regions (descending score)",
               "reprompt": "prompts inside the region only; positive + up to 4 nearest negatives within 2 x median NN distance",
               "keep": "new mask > 50% inside the region",
               "date_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
               "repo_commit": REPO_COMMIT, "heldout_accessed": False, "test_set_accessed": False}, f, indent=2)
print("Saved to:", D5)

SNP-v2: flagged masks, re-prompts and replacements per image, by stratum:


,flagged2[S],reprompted2[S],replaced2[S],new_masks2[S],flagged2[LM0],reprompted2[LM0],replaced2[LM0],new_masks2[LM0]
density_stratum,,,,,,,,
S1_0_20,1.20,7.35,1.11,6.64,0.80,5.79,0.70,5.34
S2_20_40,0.94,5.76,0.79,5.40,0.76,5.11,0.56,4.31
S3_40_60,1.18,9.38,0.88,8.42,0.92,5.58,0.72,4.47
S4_60_80,1.42,8.13,1.15,7.50,1.28,4.85,1.05,4.40
S5_80_95,1.68,6.70,1.33,5.78,1.78,8.32,1.37,7.15
S6_95_100,2.27,25.28,1.45,21.78,2.00,20.58,1.47,18.78


SNP-v2 (train_dev400) — metrics:


,system,MAE_reweighted,mean_signed,MAE_S1,MAE_S2,MAE_S3,MAE_S4,MAE_S5,MAE_S6
0,B0_v1,35.67,-62.67,5.56,7.74,13.30,22.93,59.03,338.08
1,GPV_R4_v1,29.94,-46.78,8.62,7.70,12.45,16.13,40.52,297.63
2,H1b_v1,28.65,-41.28,10.46,9.49,12.35,14.20,37.33,274.98
3,H1b[neg],36.95,-30.90,19.19,19.56,27.60,22.87,38.42,266.87
4,H1b-L[M0],28.91,-49.22,6.89,6.14,9.93,14.90,42.42,299.43
5,H1b-SN[S],31.45,-37.94,12.80,10.95,19.00,18.18,37.97,271.28
6,H1b-SN[LM0],30.47,-45.25,9.56,8.59,15.08,15.00,42.35,289.43
7,GPV_R4-SN2[S],30.49,-41.84,10.91,9.12,13.82,18.45,39.60,281.78
8,H1b-SN2[S],30.17,-40.30,11.68,10.10,14.07,17.40,38.30,275.60
9,GPV_R4-SN2[LM0],30.05,-49.52,7.78,6.79,10.85,15.87,44.52,302.35


Targets (difference [95% CI]; negative = system better):


,system,S1 vs B0,S2 vs B0,S5 vs H1b,S6 vs H1b,overall vs H1b,T1,T2',T3',ALL
0,H1b[neg],"+13.62 [+10.64, +16.70]","+11.82 [+8.64, +15.16]","+1.08 [-3.17, +5.62]","-8.12 [-12.67, -3.15]","+8.30 [+6.76, +9.91]",False,True,False,False
1,H1b-L[M0],"+1.33 [-0.44, +3.11]","-1.60 [-3.34, +0.16]","+5.08 [-0.08, +11.52]","+24.45 [+12.60, +38.02]","+0.26 [-1.03, +1.64]",True,False,False,False
2,H1b-SN[S],"+7.24 [+4.68, +10.02]","+3.21 [+0.73, +5.88]","+0.63 [-2.70, +4.43]","-3.70 [-9.48, +2.58]","+2.80 [+1.69, +3.98]",False,True,False,False
3,H1b-SN[LM0],"+4.00 [+1.35, +6.83]","+0.85 [-1.71, +3.50]","+5.02 [-0.40, +11.70]","+14.45 [+1.45, +28.95]","+1.82 [+0.37, +3.37]",False,False,False,False
4,GPV_R4-SN2[S],"+5.35 [+3.25, +7.65]","+1.39 [-0.59, +3.58]","+2.27 [-1.70, +7.92]","+6.80 [+0.10, +14.35]","+1.84 [+0.78, +3.03]",False,False,False,False
5,H1b-SN2[S],"+6.11 [+3.94, +8.41]","+2.36 [+0.10, +4.78]","+0.97 [-1.93, +4.77]","+0.62 [-5.12, +6.80]","+1.52 [+0.69, +2.47]",False,True,False,False
6,GPV_R4-SN2[LM0],"+2.21 [+0.00, +4.62]","-0.95 [-2.94, +1.19]","+7.18 [+1.02, +15.03]","+27.37 [+14.78, +41.52]","+1.40 [-0.08, +2.99]",True,False,False,False
7,H1b-SN2[LM0],"+3.01 [+0.59, +5.61]","-0.81 [-2.86, +1.39]","+4.08 [-1.03, +10.67]","+18.52 [+6.05, +32.53]","+0.62 [-0.71, +2.08]",False,False,False,False


Saved to: /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B3NS_Selective_Negative_Prompts/section5_snp_v2
